In [0]:
# ============================================================
# Notebook : 03_Data_Validation
# Layer    : Bronze -> Validation
# ============================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window

BASE_PATH = "/Volumes/banking_catalog/banking_project/banking_volume"

BRONZE_PATH = f"{BASE_PATH}/bronze/transactions"

VALID_PATH = f"{BASE_PATH}/silver/transactions"

REJECTED_PATH = f"{BASE_PATH}/rejected/transactions"

PIPELINE_NAME = "Banking Transaction Pipeline"

print("="*80)
print("Starting Data Validation")
print("="*80)

In [0]:
bronze_df = (
    spark.read
         .format("delta")
         .load(BRONZE_PATH)
)

print(f"Total Bronze Records : {bronze_df.count()}")

display(bronze_df)

In [0]:
window_spec = Window.partitionBy("transaction_id") \
                    .orderBy(col("load_timestamp").desc())

validation_df = (

    bronze_df

    .withColumn("row_num", row_number().over(window_spec))

)

In [0]:
validation_df = (

    validation_df

    .withColumn(

        "rejection_reason",

        when(col("transaction_id").isNull(),"Transaction ID Missing")

        .when(col("customer_id").isNull(),"Customer ID Missing")

        .when(col("account_number").isNull(),"Account Number Missing")

        .when(col("amount").isNull(),"Amount Missing")

        .when(col("amount") <= 0,"Invalid Amount")

        .when(col("currency").isNull(),"Currency Missing")

        .when(col("transaction_timestamp").isNull(),"Timestamp Missing")

        .when(~col("transaction_type").isin(

            "ATM_WITHDRAWAL",
            "UPI_TRANSFER",
            "NEFT",
            "RTGS",
            "CARD_PAYMENT",
            "IB_TRANSFER",
            "MB_TRANSFER",
            "CASH_DEPOSIT"

        ),"Invalid Transaction Type")

        .when(col("row_num") > 1,"Duplicate Transaction")

    )

)

In [0]:
valid_df = (

    validation_df

    .filter(col("rejection_reason").isNull())

    .drop("row_num","rejection_reason")

)

invalid_df = (

    validation_df

    .filter(col("rejection_reason").isNotNull())

    .drop("row_num")

)

In [0]:
valid_count = valid_df.count()

invalid_count = invalid_df.count()

print("="*80)

print(f"Valid Records   : {valid_count}")

print(f"Invalid Records : {invalid_count}")

print("="*80)

In [0]:
(

    valid_df

    .write

    .format("delta")

    .mode("overwrite")

    .save(VALID_PATH)

)

print("Valid Records Saved")

In [0]:
(

    invalid_df

    .write

    .format("delta")

    .mode("overwrite")

    .save(REJECTED_PATH)

)

print("Rejected Records Saved")

In [0]:
# Create or replace tables with data from Volume paths
spark.sql(f"""
CREATE OR REPLACE TABLE banking_catalog.banking_project.silver_transactions
USING DELTA
AS SELECT * FROM delta.`{VALID_PATH}`
""")

spark.sql(f"""
CREATE OR REPLACE TABLE banking_catalog.banking_project.rejected_transactions
USING DELTA
AS SELECT * FROM delta.`{REJECTED_PATH}`
""")

print("Delta tables created successfully")

In [0]:
print("="*80)

print("Validation Completed Successfully")

print(f"Total Records     : {bronze_df.count()}")

print(f"Valid Records     : {valid_count}")

print(f"Rejected Records  : {invalid_count}")

print("="*80)